In [0]:
silver_df = spark.table("workspace.nyc_taxi.silver_trips")

In [0]:
from pyspark.sql.functions import *

In [0]:
daily_summary = (
    silver_df
    .groupBy(
        to_date("tpep_pickup_datetime").alias("trip_date")
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("total_revenue"),

        avg("fare_amount").alias("average_fare"),

        avg("trip_distance").alias("average_distance"),

        avg("tip_amount").alias("average_tip")
    )
)

In [0]:
display(daily_summary)

trip_date,total_trips,total_revenue,average_fare,average_distance,average_tip
2024-02-24,116166,2940419.9600000028,17.57297832412233,2.9189423755660084,3.003461253723137
2002-12-31,3,53.65,12.799999999999999,2.4566666666666666,1.0
2024-01-24,103903,2818198.7000000053,18.1256743308663,2.959606171140395,3.4688884825269986
2024-02-18,92220,2472821.470000016,18.72238700932557,3.3326019301669936,3.1040775319887364
2024-02-21,102266,2785133.050000009,18.239754463849195,3.070632077132185,3.4225916726967105
2024-01-07,66559,1925061.0200000128,20.077106476960285,3.8796950074370113,3.549625895821767
2024-01-23,98683,2663781.050000011,18.110350516299636,2.9735490408682397,3.403288915010704
2024-03-03,95487,2726702.8500000113,20.01658854084845,3.6105968351712696,3.39060081477062
2024-01-01,79701,2475436.960000014,22.48463381889808,4.056878834644487,3.3152754670581506
2024-01-05,101847,2760208.4400000107,18.308837079148155,3.181356839180338,3.272009484815474


In [0]:
daily_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.nyc_taxi.gold_daily_summary"
)

In [0]:
hourly_summary = (
    silver_df
    .groupBy(
        hour("tpep_pickup_datetime")
        .alias("pickup_hour")
    )
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("revenue")
    )
)

In [0]:
display(hourly_summary)

pickup_hour,total_trips,revenue
11,457388,1.1885857889999835E7
22,492847,1.3773548289999397E7
10,424166,1.1148550879999943E7
20,543246,1.4677435519999282E7
9,400865,1.0400832309999894E7
0,264218,7399673.919999828
1,177701,4495739.529999981
21,543031,1.4686371809999432E7
8,367509,9376903.6799999
2,116768,2828575.740000015


In [0]:
hourly_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.nyc_taxi.gold_hourly_summary"
)

In [0]:
payment_summary = (
    silver_df
    .groupBy("payment_type")
    .agg(
        count("*").alias("trips"),

        sum("total_amount").alias("revenue")
    )
)

In [0]:
payment_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.nyc_taxi.gold_payment_summary")

In [0]:
vendor_summary = (
    silver_df
    .groupBy("VendorID")
    .agg(
        count("*").alias("total_trips"),

        sum("total_amount").alias("revenue"),

        avg("fare_amount").alias("average_fare")
    )
)

In [0]:
vendor_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.nyc_taxi.gold_vendor_summary")

In [0]:
pickup_summary = (
    silver_df
    .groupBy("PULocationID")
    .agg(
        count("*").alias("pickup_count")
    )
    .orderBy(
        desc("pickup_count")
    )
)

In [0]:
pickup_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.nyc_taxi.gold_pickup_summary")